In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

from typing import Dict

import pandas as pd
pd.set_option("display.max_columns",30)
import requests

In [2]:

"""
Steps for the Transform Load Lambda function
    
1. Get data from S3 (taxi, weather)
2. Weather data transformations - DONE
3. Taxi data transformations - DONE
4. Update dim_payment_type - DONE
5. Update dim_company - DONE
6. Update fact_taxi_trips with the ids from dim_payment_type and dim_company - DONE
7. Upload dim_weather to S3
8. Upload fact_taxi_trips to S3
9. Upload dim_payment_type and dim_company (current, and previous version)

"""


'\nSteps for the Transform Load Lambda function\n\n1. Get data from S3 (taxi, weather)\n2. Weather data transformations - DONE\n3. Taxi data transformations - DONE\n4. Update dim_payment_type - DONE\n5. Update dim_company - DONE\n6. Update fact_taxi_trips with the ids from dim_payment_type and dim_company - DONE\n7. Upload dim_weather to S3\n8. Upload fact_taxi_trips to S3\n9. Upload dim_payment_type and dim_company (current, and previous version)\n\n'

In [3]:
current_datetime = datetime.now() - relativedelta(month=2)
formatted_datetime = current_datetime.strftime("%Y-%m-%d")

url = (

    f"https://data.cityofchicago.org/resource/ajtu-isnz.json?"
    f"$where=trip_start_timestamp >= '{formatted_datetime}T00:00:00' "
    f"AND trip_start_timestamp <= '{formatted_datetime}T23:59:59' "
    f"&$limit=30000"
)

response = requests.get(url)
data = response.json()

taxi_trips = pd.DataFrame(data)

taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,000374dcbfb7ccf6e0abe3b0021e1da12786aa58,179f1a051e9e6d3fc0726628962faff68506086ee8df14...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,1532,14.84,76,7,37.5,8.4,0,4,50.4,Credit Card,Blue Ribbon Taxi Association,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.922686284,-87.649488729,"{'type': 'Point', 'coordinates': [-87.64948872...",NaN,NaN
1,f47ffeda3b022b9694095aade874328d8b55f2cb,fe43642f2a2fb98dc45bea2b95638dc0c15612e66f229c...,2026-02-24T23:45:00.000,2026-02-25T00:30:00.000,2607,22.62,76,1,57,2,0,4,63.5,Credit Card,5 Star Taxi,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",42.009622881,-87.670166857,"{'type': 'Point', 'coordinates': [-87.67016685...",NaN,NaN
2,e4d07301a6705c1758eb0cb43abc9476c013b2c5,a7aaa6374b9f88b5fd31a1106378dffccb77e1261bc57a...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,1328,10.46,14,34,28.5,0,0,0,28.5,Prcard,5 Star Taxi,41.968069,-87.721559063,"{'type': 'Point', 'coordinates': [-87.72155906...",41.842076117,-87.633973422,"{'type': 'Point', 'coordinates': [-87.63397342...",NaN,NaN
3,e3b4fdb3af0276e16f4c8f5ebfac53be88b7ea97,a692055914b912499c7042ab77631e9da33065435edbde...,2026-02-24T23:45:00.000,2026-02-25T00:00:00.000,922,11.58,24,10,29.75,0,0,0,29.75,Prcard,Flash Cab,41.901206994,-87.676355989,"{'type': 'Point', 'coordinates': [-87.67635598...",41.985015101,-87.804532006,"{'type': 'Point', 'coordinates': [-87.80453200...",NaN,NaN
4,e250417b60245c8ae60c5ea54314072f69818046,0734b10e5be7c6aa0f018e381095204a81011ef78493ba...,2026-02-24T23:45:00.000,2026-02-25T00:15:00.000,1588,14.12,76,3,36,10.12,0,4,50.62,Credit Card,City Service,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.96581197,-87.655878786,"{'type': 'Point', 'coordinates': [-87.65587878...",NaN,NaN


### Taxi data transformations

In [4]:
def taxi_trips_transformations(taxi_trips: pd.DataFrame) -> pd.DataFrame:
    
    """Perform transformations on the taxi data

    1. Drop selected columns.
    2. Drop NULL values across all columns.
    3. Rename selected columns.
    4. Create "datetime_for_weather" helper column(for dim_weather join).
    
    :param taxi_trips:  The DataFrame holding the daily taxi trips.
    :raise TypeError:   When taxi_trips parameter is not a valid pandas DataFrame.
    :return:            Transformed taxi trips DataFrame.
    
    """
    if not isinstance (taxi_trips, pd.DataFrame):
        raise TypeError("taxi_trips is not a valid pandas DtaFrame.")
    taxi_trips.drop(["pickup_census_tract","dropoff_census_tract",
                    "pickup_centroid_location","dropoff_centroid_location"],axis=1,inplace=True)

    taxi_trips.dropna(inplace=True)

    taxi_trips.rename(columns= { "pickup_community_area" : "pickup_community_area_id", 
                            "dropoff_community_area" : "dropoff_community_area_id"
                            }, inplace=True)

    taxi_trips["trip_start_timestamp"] = pd.to_datetime(taxi_trips["trip_start_timestamp"])

    taxi_trips["datetime_for_weather"] = taxi_trips["trip_start_timestamp"].dt.floor("h")

    return taxi_trips

In [5]:
taxi_trips_transformed = taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()


,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,datetime_for_weather
0,000374dcbfb7ccf6e0abe3b0021e1da12786aa58,179f1a051e9e6d3fc0726628962faff68506086ee8df14...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,1532,14.84,76,7,37.5,8.4,0,4,50.4,Credit Card,Blue Ribbon Taxi Association,41.980264315,-87.913624596,41.922686284,-87.649488729,2026-02-24 23:00:00
1,f47ffeda3b022b9694095aade874328d8b55f2cb,fe43642f2a2fb98dc45bea2b95638dc0c15612e66f229c...,2026-02-24 23:45:00,2026-02-25T00:30:00.000,2607,22.62,76,1,57,2,0,4,63.5,Credit Card,5 Star Taxi,41.980264315,-87.913624596,42.009622881,-87.670166857,2026-02-24 23:00:00
2,e4d07301a6705c1758eb0cb43abc9476c013b2c5,a7aaa6374b9f88b5fd31a1106378dffccb77e1261bc57a...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,1328,10.46,14,34,28.5,0,0,0,28.5,Prcard,5 Star Taxi,41.968069,-87.721559063,41.842076117,-87.633973422,2026-02-24 23:00:00
3,e3b4fdb3af0276e16f4c8f5ebfac53be88b7ea97,a692055914b912499c7042ab77631e9da33065435edbde...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,922,11.58,24,10,29.75,0,0,0,29.75,Prcard,Flash Cab,41.901206994,-87.676355989,41.985015101,-87.804532006,2026-02-24 23:00:00
4,e250417b60245c8ae60c5ea54314072f69818046,0734b10e5be7c6aa0f018e381095204a81011ef78493ba...,2026-02-24 23:45:00,2026-02-25T00:15:00.000,1588,14.12,76,3,36,10.12,0,4,50.62,Credit Card,City Service,41.980264315,-87.913624596,41.96581197,-87.655878786,2026-02-24 23:00:00


In [6]:
taxi_trips_transformed.info()

<class 'pandas.DataFrame'>
Index: 15468 entries, 0 to 17102
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     15468 non-null  str           
 1   taxi_id                     15468 non-null  str           
 2   trip_start_timestamp        15468 non-null  datetime64[us]
 3   trip_end_timestamp          15468 non-null  str           
 4   trip_seconds                15468 non-null  str           
 5   trip_miles                  15468 non-null  str           
 6   pickup_community_area_id    15468 non-null  str           
 7   dropoff_community_area_id   15468 non-null  str           
 8   fare                        15468 non-null  str           
 9   tips                        15468 non-null  str           
 10  tolls                       15468 non-null  str           
 11  extras                      15468 non-null  str           
 12  trip_t

#### Dim company and dim Payment type update 

In [ ]:
def update_dim_company_dim_payment_type (taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, id_col: str, value_col: str) -> pd.DataFrame:

    """Extend the dimension DataFrame with new values if there are any

    :param taxi_trips:      DataFrame with the daily taxi trips
    :param dim_df:          DataFrame with the dimension data(company, payment_type)
    :param id_col:          The id columns of the dimension DataFrame.
    :param value_col:       Name of the column in dimension DataFrame containing the values.
    :return:                The updated dimension DataFrame, if new values are in the taxi data, they will be loaded to it.

    """

    todays_dim_data = pd.DataFrame(taxi_trips[value_col].unique(), columns=[value_col])
    new_dim_data = todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    if not new_dim_data.empty:
        max_id = dim_df[id_col].max()
        new_dim_data[id_col] = range(max_id + 1, max_id +1 + len(new_dim_data))
        dim_df = pd.concat([dim_df, new_dim_data], ignore_index=True)

    return dim_df

In [7]:
dim_payment_type = taxi_trips["payment_type"].drop_duplicates().reset_index(drop=True)

dim_payment_type = pd.DataFrame(

    {
        "payment_type_id": range(1,len(dim_payment_type) + 1),
        "payment_type": dim_payment_type
    }

)

dim_company = taxi_trips["company"].drop_duplicates().reset_index(drop=True)

dim_company = pd.DataFrame(

    {
        "company_id": range(1,len(dim_company) + 1),
        "company": dim_company
    }

)


In [8]:
dim_payment_type_updated = update_dim_company_dim_payment_type(taxi_trips, dim_payment_type, "payment_type_id", "payment_type")

dim_company_updated = update_dim_company_dim_payment_type(taxi_trips, dim_company, "company_id", "company")

In [20]:
dim_payment_type_updated

,payment_type_id,payment_type
0,1,Credit Card
1,2,Prcard
2,3,Cash
3,4,Mobile
4,5,Unknown
5,6,No Charge
6,7,Dispute


In [21]:
dim_company_updated

,company_id,company
0,1,Blue Ribbon Taxi Association
1,2,5 Star Taxi
2,3,Flash Cab
3,4,City Service
4,5,Globe Taxi
5,6,Chicago Independents
6,7,Taxicab Insurance Agency Llc
7,8,Sun Taxi
8,9,Medallion Leasin
9,10,Wolley Taxi


In [26]:
dummy_payment_type_data = [
    {"payment_type":"Credit Card"},
    {"payment_type":"X"},
    {"payment_type":"Z"},
    {"payment_type":"Z"},
    
]

dummy_payment_type_data_df = pd.DataFrame(dummy_payment_type_data)

dummy_company_data = [
    {"company":"Tac - Yellow Non Color"},
    {"company":"TBC - Black Non Color"},
    {"company":"TBC - Blue Non Color"},
    {"company":"TBC - Blue Non Color"},
    
]

dummy_company_data_df = pd.DataFrame(dummy_company_data)



In [27]:
dim_payment_type_updated = update_dim_company_dim_payment_type(dummy_payment_type_data_df, dim_payment_type, "payment_type_id", "payment_type")

dim_company_updated = update_dim_company_dim_payment_type(dummy_company_data_df, dim_company, "company_id", "company")

In [28]:
dim_payment_type_updated

,payment_type_id,payment_type
0,1,Credit Card
1,2,Prcard
2,3,Cash
3,4,Mobile
4,5,Unknown
5,6,No Charge
6,7,Dispute
7,8,X
8,9,Z


In [29]:
dim_company_updated

,company_id,company
0,1,Blue Ribbon Taxi Association
1,2,5 Star Taxi
2,3,Flash Cab
3,4,City Service
4,5,Globe Taxi
5,6,Chicago Independents
6,7,Taxicab Insurance Agency Llc
7,8,Sun Taxi
8,9,Medallion Leasin
9,10,Wolley Taxi


#### Update fact_taxi_trips based on company and payment type ids

In [9]:
def update_fact_taxi_trips_with_dimension_data (taxi_trips: pd.DataFrame, dim_payment_type: pd.DataFrame, dim_company: pd.DataFrame) -> pd.DataFrame:
   
    """Update the fact_taxi_trips with the dim_company master and dim_payment_type master ids, and deleted the duplicated columns

    :param taxi_trips:              The DataFrame with the daily taxi trips.
    :param dim_payment_type:        The payment_type dimension table.
    :param dim_company_type:        The company dimension table.
    :return:                        The taxi_trips data, with only payment_type_id and company_id, without company or
                                    payment_type values.

    """
    
    fact_taxi_trips = taxi_trips.merge(dim_payment_type, on="payment_type")
    fact_taxi_trips = fact_taxi_trips.merge(dim_company, on="company")
    fact_taxi_trips.drop(["payment_type","company"], axis = 1, inplace=True)

    return fact_taxi_trips

In [10]:
taxi_trips_transformed_with__dim_ids = update_fact_taxi_trips_with_dimension_data(taxi_trips_transformed,dim_payment_type, dim_company)
taxi_trips_transformed_with__dim_ids.head()


,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,datetime_for_weather,payment_type_id,company_id
0,000374dcbfb7ccf6e0abe3b0021e1da12786aa58,179f1a051e9e6d3fc0726628962faff68506086ee8df14...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,1532,14.84,76,7,37.5,8.4,0,4,50.4,41.980264315,-87.913624596,41.922686284,-87.649488729,2026-02-24 23:00:00,1,1
1,f47ffeda3b022b9694095aade874328d8b55f2cb,fe43642f2a2fb98dc45bea2b95638dc0c15612e66f229c...,2026-02-24 23:45:00,2026-02-25T00:30:00.000,2607,22.62,76,1,57,2,0,4,63.5,41.980264315,-87.913624596,42.009622881,-87.670166857,2026-02-24 23:00:00,1,2
2,e4d07301a6705c1758eb0cb43abc9476c013b2c5,a7aaa6374b9f88b5fd31a1106378dffccb77e1261bc57a...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,1328,10.46,14,34,28.5,0,0,0,28.5,41.968069,-87.721559063,41.842076117,-87.633973422,2026-02-24 23:00:00,2,2
3,e3b4fdb3af0276e16f4c8f5ebfac53be88b7ea97,a692055914b912499c7042ab77631e9da33065435edbde...,2026-02-24 23:45:00,2026-02-25T00:00:00.000,922,11.58,24,10,29.75,0,0,0,29.75,41.901206994,-87.676355989,41.985015101,-87.804532006,2026-02-24 23:00:00,2,3
4,e250417b60245c8ae60c5ea54314072f69818046,0734b10e5be7c6aa0f018e381095204a81011ef78493ba...,2026-02-24 23:45:00,2026-02-25T00:15:00.000,1588,14.12,76,3,36,10.12,0,4,50.62,41.980264315,-87.913624596,41.96581197,-87.655878786,2026-02-24 23:00:00,1,4


#### Weather transformations

In [ ]:
def transform_weather(weather_data: Dict) -> pd.DataFrame: 
    """Select and transform weather data

    :param weather_data:    The daily weather data from the Open Meteo API.
    :return:                Transformed weather pandas DataFrame.
    """
     
    weather_data = {
        "datetime": data["hourly"]["time"],
        "temperature": data["hourly"]["temperature_2m"],
        "wind_speed": data["hourly"]["wind_speed_10m"],
        "rain": data["hourly"]["rain"],
        "precipitation": data["hourly"]["precipitation"],
    }

    weather_df = pd.DataFrame(weather_data)

    weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

    return weather_df

In [18]:
current_datetime = datetime.now() - relativedelta(month=2)
formatted_datetime = current_datetime.strftime("%Y-%m-%d")

url = ("https://archive-api.open-meteo.com/v1/era5")

params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": formatted_datetime,
    "end_date": formatted_datetime,
    "hourly":"temperature_2m,wind_speed_10m,rain,precipitation"

}

response = requests.get(url, params = params)
weather_raw_data = response.json()

weather_raw_data

{'latitude': 41.862915,
 'longitude': -87.64877,
 'generationtime_ms': 0.7028579711914062,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 179.0,
 'hourly_units': {'time': 'iso8601',
  'temperature_2m': '°C',
  'wind_speed_10m': 'km/h',
  'rain': 'mm',
  'precipitation': 'mm'},
 'hourly': {'time': ['2026-02-24T00:00',
   '2026-02-24T01:00',
   '2026-02-24T02:00',
   '2026-02-24T03:00',
   '2026-02-24T04:00',
   '2026-02-24T05:00',
   '2026-02-24T06:00',
   '2026-02-24T07:00',
   '2026-02-24T08:00',
   '2026-02-24T09:00',
   '2026-02-24T10:00',
   '2026-02-24T11:00',
   '2026-02-24T12:00',
   '2026-02-24T13:00',
   '2026-02-24T14:00',
   '2026-02-24T15:00',
   '2026-02-24T16:00',
   '2026-02-24T17:00',
   '2026-02-24T18:00',
   '2026-02-24T19:00',
   '2026-02-24T20:00',
   '2026-02-24T21:00',
   '2026-02-24T22:00',
   '2026-02-24T23:00'],
  'temperature_2m': [-4.4,
   -4.9,
   -5.4,
   -5.4,
   -5.8,
   -5.6,
   -6.0,
   -6.5,
   -6.9,
   -7.

In [19]:
weather_df = transform_weather(weather_raw_data)

weather_df.head()

TypeError: list indices must be integers or slices, not str